In [7]:
# === EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===
from pathlib import Path
import csv
import json
import shutil

LAST_NAME = "Viana"
STUDENT_ID = "TUPM-26-1953"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

DATA_DIR = Path("../data")
OUTPUT_DIR = Path(".")
BACKUP_DIR = Path("../backup")

input_path = DATA_DIR / "user_behavior_dataset.csv"
output_csv = OUTPUT_DIR / f"{LAST_NAME}_mobile_usage.csv"
summary_json = OUTPUT_DIR / f"{LAST_NAME}_mobile_summary.json"
backup_path = BACKUP_DIR / output_csv.name

# Rules
OS_TARGET = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

total_records = 0
os_matches = 0
behavior_matches = 0
battery_values = []
data_values = []
selected_rows = []

with open(input_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        if row["Operating System"].strip() == OS_TARGET:
            os_matches += 1
            try:
                if int(row["User Behavior Class"]) == BEHAVIOR_CLASS:
                    behavior_matches += 1
                    battery = float(row["Battery Drain (mAh/day)"])
                    data_mb = float(row["Data Usage (MB/day)"])
                    battery_values.append(battery)
                    data_values.append(data_mb)
                    selected_rows.append({
                        "User ID": row["User ID"],
                        "Device Model": row["Device Model"],
                        "Operating System": row["Operating System"],
                        "Battery Drain (mAh/day)": battery,
                        "Data Usage (MB/day)": data_mb
                    })
            except (ValueError, KeyError):
                continue

# Calculate averages
avg_battery = sum(battery_values) / len(battery_values) if battery_values else 0
avg_data = sum(data_values) / len(data_values) if data_values else 0

# Export CSV
with open(output_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["User ID", "Device Model", "Operating System",
                                              "Battery Drain (mAh/day)", "Data Usage (MB/day)"])
    writer.writeheader()
    writer.writerows(selected_rows)

# Export JSON summary
summary = {
    "operating_system_and_behavior_class": f"{OS_TARGET}, Class {BEHAVIOR_CLASS}",
    "total_records_in_dataset": total_records,
    "records_matching_operating_system": os_matches,
    "records_in_target_behavior_class": behavior_matches,
    "average_battery_drain_mAh_per_day": round(avg_battery, 2),
    "average_data_usage_MB_per_day": round(avg_data, 2),
    "count_verification": {
        "selected_from_csv": len(selected_rows),
        "reported_count": behavior_matches,
        "match": len(selected_rows) == behavior_matches
    }
}
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

# Backup
shutil.copy2(output_csv, backup_path)
backup_size = backup_path.stat().st_size

# Output
print("=== EXERCISE 2 RESULTS ===")
print(f"Operating System & Behavior Class: {OS_TARGET}, Class {BEHAVIOR_CLASS}")
print(f"Total records in dataset: {total_records}")
print(f"Records matching OS: {os_matches}")
print(f"Records in target class: {behavior_matches}")
print(f"Average Battery Drain: {avg_battery:.2f} mAh/day")
print(f"Average Data Usage: {avg_data:.2f} MB/day")
print(f"Count verification: Exported {len(selected_rows)} of {behavior_matches} — { 'MATCH ✓' if len(selected_rows)==behavior_matches else 'MISMATCH ✗' }")
print(f"Exported CSV file: {output_csv.name}")
print(f"JSON summary file: {summary_json.name}")
print(f"Backup file & size: {backup_path.name} — {backup_size} bytes")

=== EXERCISE 2 RESULTS ===
Operating System & Behavior Class: iOS, Class 4
Total records in dataset: 700
Records matching OS: 146
Records in target class: 29
Average Battery Drain: 2123.21 mAh/day
Average Data Usage: 1221.45 MB/day
Count verification: Exported 29 of 29 — MATCH ✓
Exported CSV file: Viana_mobile_usage.csv
JSON summary file: Viana_mobile_summary.json
Backup file & size: Viana_mobile_usage.csv — 1037 bytes
